# V3 Prompt Evaluation Results

Compare Human A vs Human B first to estimate the human agreement ceiling, then compare the V3 LLM annotations against each human rater and average those two LLM-vs-human scores.

Primary metrics:
- F1 Score over flattened `(student, problem, KC)` binary decisions.
- Jaccard Value as mean per-problem set Jaccard, matching the prior Jaccard analysis style.
- Cohen's Kappa over flattened binary KC decisions.
- Gwet's AC1 over flattened binary KC decisions.

In [37]:
import json
import math
import sys
from pathlib import Path

import pandas as pd

ROOT = Path('/mnt/d/Projects/kintsugi')
sys.path.insert(0, str(ROOT))

HUMAN_DIR = ROOT / 'dataset' / 'Rater_KC_Tags' / 'Rated_KC_V3'
LLM_DIR = ROOT / 'results' / 'human_validation' / 'llm_v3_10students'
OUTPUT_DIR = ROOT / 'results' / 'human_validation' / 'v3_prompt_eval_results'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

STUDENT_IDS = ['10155', '9948', '14189', '14352', '14362', '14363', '14374', '14414', '14474', '14499']

KC_COLUMNS = [
    'If/Else', 'NestedIf', 'While', 'For', 'NestedFor',
    'Math+-*/', 'Math%', 'LogicAndNotOr', 'LogicCompareNum', 'LogicBoolean',
    'StringFormat', 'StringConcat', 'StringIndex', 'StringLen',
    'StringEqual', 'CharEqual', 'ArrayIndex', 'DefFunction'
]
VALID_KCS = set(KC_COLUMNS)

# Prior exp17 skipped problems where both raters had no gaps for set-overlap Jaccard.
INCLUDE_EMPTY_JACCARD = False

print(f'Human annotations: {HUMAN_DIR}')
print(f'LLM annotations: {LLM_DIR}')
print(f'Output dir: {OUTPUT_DIR}')

Human annotations: /mnt/d/Projects/kintsugi/dataset/Rater_KC_Tags/Rated_KC_V3
LLM annotations: /mnt/d/Projects/kintsugi/results/human_validation/llm_v3_10students
Output dir: /mnt/d/Projects/kintsugi/results/human_validation/v3_prompt_eval_results


In [38]:
def find_one(pattern: str, directory: Path) -> Path:
    matches = sorted(directory.glob(pattern))
    if not matches:
        raise FileNotFoundError(f'No file matched {pattern} in {directory}')
    if len(matches) > 1:
        print(f'WARNING: multiple matches for {pattern}; using {matches[-1].name}')
    return matches[-1]


def normalize_gaps(value) -> set[str]:
    if isinstance(value, dict):
        gaps = value.get('gaps', [])
    elif isinstance(value, list):
        gaps = value
    else:
        gaps = []
    if not isinstance(gaps, list):
        return set()
    return {gap for gap in gaps if gap in VALID_KCS}


def load_annotation_file(path: Path) -> tuple[str, str, dict[str, set[str]]]:
    with path.open('r', encoding='utf-8') as f:
        data = json.load(f)
    rater = data.get('rater', path.stem)
    student_id = str(data.get('studentId', data.get('student_id', 'unknown')))
    annotations = {}
    for pid, value in data.get('annotations', {}).items():
        annotations[f'{student_id}_{pid}'] = normalize_gaps(value)
    return rater, student_id, annotations


def merge_student_files(file_map: dict[str, Path], label: str) -> dict[str, set[str]]:
    merged = {}
    for sid, path in file_map.items():
        _, loaded_sid, anns = load_annotation_file(path)
        if loaded_sid != sid:
            print(f'WARNING: expected student {sid}, found {loaded_sid} in {path.name}')
        merged.update(anns)
    print(f'{label}: loaded {len(file_map)} files, {len(merged)} student-problem annotations')
    return merged

In [39]:
human_a_files = {
    sid: find_one(f'kc_annotations_Pranay Ghuge_{sid}_*.json', HUMAN_DIR)
    for sid in STUDENT_IDS
}
human_b_files = {
    sid: find_one(f'kc_annotations_Arundhati Das_{sid}_*.json', HUMAN_DIR)
    for sid in STUDENT_IDS
}
llm_files = {
    sid: find_one(f'llm_v3_annotations_{sid}.json', LLM_DIR)
    for sid in STUDENT_IDS
}

human_a = merge_student_files(human_a_files, 'Human A')
human_b = merge_student_files(human_b_files, 'Human B')
llm_v3 = merge_student_files(llm_files, 'LLM V3')

common_items = sorted(set(human_a) & set(human_b) & set(llm_v3), key=lambda x: (int(x.split('_')[0]), int(x.split('_')[1])))
print(f'Common student-problem items across all three raters: {len(common_items)}')

coverage_df = pd.DataFrame([
    {'Rater': 'Human A', 'Files': len(human_a_files), 'Annotations': len(human_a)},
    {'Rater': 'Human B', 'Files': len(human_b_files), 'Annotations': len(human_b)},
    {'Rater': 'LLM V3', 'Files': len(llm_files), 'Annotations': len(llm_v3)},
])
coverage_df

Human A: loaded 10 files, 372 student-problem annotations
Human B: loaded 10 files, 372 student-problem annotations
LLM V3: loaded 10 files, 372 student-problem annotations
Common student-problem items across all three raters: 372


,Rater,Files,Annotations
0,Human A,10,372
1,Human B,10,372
2,LLM V3,10,372


In [40]:
def binary_confusion(anns_a: dict[str, set[str]], anns_b: dict[str, set[str]], items: list[str]):
    tp = fp = fn = tn = 0
    for item in items:
        gaps_a = anns_a.get(item, set())
        gaps_b = anns_b.get(item, set())
        for kc in KC_COLUMNS:
            a = kc in gaps_a
            b = kc in gaps_b
            if a and b:
                tp += 1
            elif not a and b:
                fp += 1
            elif a and not b:
                fn += 1
            else:
                tn += 1
    return tp, fp, fn, tn


def safe_div(num: float, den: float) -> float:
    return num / den if den else 0.0


def cohen_kappa_from_counts(tp: int, fp: int, fn: int, tn: int) -> float:
    n = tp + fp + fn + tn
    if n == 0:
        return 0.0
    observed = (tp + tn) / n
    p_yes_a = (tp + fn) / n
    p_yes_b = (tp + fp) / n
    p_no_a = (fp + tn) / n
    p_no_b = (fn + tn) / n
    expected = (p_yes_a * p_yes_b) + (p_no_a * p_no_b)
    return safe_div(observed - expected, 1 - expected)


def gwet_ac1_from_counts(tp: int, fp: int, fn: int, tn: int) -> float:
    n = tp + fp + fn + tn
    if n == 0:
        return 0.0
    observed = (tp + tn) / n
    p_yes = ((tp + fn) + (tp + fp)) / (2 * n)
    p_no = 1 - p_yes
    expected = (p_yes * p_no) + (p_no * p_yes)
    return safe_div(observed - expected, 1 - expected)


def mean_problem_jaccard(anns_a: dict[str, set[str]], anns_b: dict[str, set[str]], items: list[str]) -> tuple[float, int]:
    values = []
    for item in items:
        gaps_a = anns_a.get(item, set())
        gaps_b = anns_b.get(item, set())
        union = gaps_a | gaps_b
        if not union:
            if INCLUDE_EMPTY_JACCARD:
                values.append(1.0)
            continue
        values.append(len(gaps_a & gaps_b) / len(union))
    return (sum(values) / len(values) if values else 0.0), len(values)


def pair_metrics(name_a: str, anns_a: dict[str, set[str]], name_b: str, anns_b: dict[str, set[str]], items: list[str]) -> dict:
    tp, fp, fn, tn = binary_confusion(anns_a, anns_b, items)
    precision = safe_div(tp, tp + fp)
    recall = safe_div(tp, tp + fn)
    f1 = safe_div(2 * tp, (2 * tp) + fp + fn)
    binary_jaccard = safe_div(tp, tp + fp + fn)
    problem_jaccard, jaccard_n = mean_problem_jaccard(anns_a, anns_b, items)
    return {
        'Pair': f'{name_a} vs {name_b}',
        'N Problems': len(items),
        'N KC Decisions': len(items) * len(KC_COLUMNS),
        'F1 Score': f1,
        'Jaccard Value': problem_jaccard,
        'Jaccard N': jaccard_n,
        'Binary Jaccard': binary_jaccard,
        'Kappa': cohen_kappa_from_counts(tp, fp, fn, tn),
        "Gwet AC1": gwet_ac1_from_counts(tp, fp, fn, tn),
        'Precision': precision,
        'Recall': recall,
        'TP': tp,
        'FP': fp,
        'FN': fn,
        'TN': tn,
    }


def metric_average(rows: list[dict], pair_name: str) -> dict:
    metric_cols = ['F1 Score', 'Jaccard Value', 'Binary Jaccard', 'Kappa', "Gwet AC1", 'Precision', 'Recall']
    out = {'Pair': pair_name}
    for col in metric_cols:
        out[col] = sum(row[col] for row in rows) / len(rows) if rows else 0.0
    out['N Problems'] = rows[0]['N Problems'] if rows else 0
    out['N KC Decisions'] = rows[0]['N KC Decisions'] if rows else 0
    out['Jaccard N'] = sum(row['Jaccard N'] for row in rows) / len(rows) if rows else 0
    return out

In [41]:
human_ceiling = pair_metrics('Human A', human_a, 'Human B', human_b, common_items)
llm_vs_a = pair_metrics('Human A', human_a, 'LLM V3', llm_v3, common_items)
llm_vs_b = pair_metrics('Human B', human_b, 'LLM V3', llm_v3, common_items)
llm_avg = metric_average([llm_vs_a, llm_vs_b], 'LLM V3 average vs humans')

overall_df = pd.DataFrame([human_ceiling, llm_vs_a, llm_vs_b, llm_avg])
display_cols = ['Pair', 'N Problems', 'N KC Decisions', 'F1 Score', 'Jaccard Value', 'Kappa', "Gwet AC1", 'Precision', 'Recall', 'Jaccard N']
overall_df[display_cols].style.format({
    'F1 Score': '{:.3f}',
    'Jaccard Value': '{:.3f}',
    'Kappa': '{:.3f}',
    "Gwet AC1": '{:.3f}',
    'Precision': '{:.3f}',
    'Recall': '{:.3f}',
    'Jaccard N': '{:.1f}',
})

,Pair,N Problems,N KC Decisions,F1 Score,Jaccard Value,Kappa,Gwet AC1,Precision,Recall,Jaccard N
0,Human A vs Human B,372,6696,0.758,0.668,0.745,0.971,0.785,0.734,130.0
1,Human A vs LLM V3,372,6696,0.600,0.480,0.578,0.954,0.643,0.562,131.0
2,Human B vs LLM V3,372,6696,0.637,0.523,0.618,0.960,0.658,0.616,134.0
3,LLM V3 average vs humans,372,6696,0.618,0.501,0.598,0.957,0.651,0.589,132.5


In [42]:
ceiling_rows = []
for metric in ['F1 Score', 'Jaccard Value', 'Kappa', "Gwet AC1"]:
    ceiling = human_ceiling[metric]
    llm_score = llm_avg[metric]
    ceiling_rows.append({
        'Metric': metric,
        'Human Ceiling': ceiling,
        'LLM Avg vs Humans': llm_score,
        'Gap to Ceiling': ceiling - llm_score,
        'Percent of Ceiling': safe_div(llm_score, ceiling) if ceiling > 0 else math.nan,
    })

ceiling_df = pd.DataFrame(ceiling_rows)
ceiling_df.style.format({
    'Human Ceiling': '{:.3f}',
    'LLM Avg vs Humans': '{:.3f}',
    'Gap to Ceiling': '{:.3f}',
    'Percent of Ceiling': '{:.1%}',
})

,Metric,Human Ceiling,LLM Avg vs Humans,Gap to Ceiling,Percent of Ceiling
0,F1 Score,0.758,0.618,0.140,81.5%
1,Jaccard Value,0.668,0.501,0.167,75.1%
2,Kappa,0.745,0.598,0.147,80.3%
3,Gwet AC1,0.971,0.957,0.014,98.5%


In [43]:
student_rows = []
for sid in STUDENT_IDS:
    sid_items = [item for item in common_items if item.startswith(f'{sid}_')]
    h_ceiling = pair_metrics('Human A', human_a, 'Human B', human_b, sid_items)
    l_a = pair_metrics('Human A', human_a, 'LLM V3', llm_v3, sid_items)
    l_b = pair_metrics('Human B', human_b, 'LLM V3', llm_v3, sid_items)
    l_avg = metric_average([l_a, l_b], 'LLM V3 average vs humans')
    row = {
        'StudentID': sid,
        'N Problems': len(sid_items),
    }
    for metric in ['F1 Score', 'Jaccard Value', 'Kappa', "Gwet AC1"]:
        row[f'Ceiling {metric}'] = h_ceiling[metric]
        row[f'LLM Avg {metric}'] = l_avg[metric]
        row[f'Percent Ceiling {metric}'] = safe_div(l_avg[metric], h_ceiling[metric]) if h_ceiling[metric] > 0 else math.nan
    student_rows.append(row)

student_df = pd.DataFrame(student_rows)
student_df.style.format({col: '{:.3f}' for col in student_df.columns if col not in ['StudentID', 'N Problems']})

,StudentID,N Problems,Ceiling F1 Score,LLM Avg F1 Score,Percent Ceiling F1 Score,Ceiling Jaccard Value,LLM Avg Jaccard Value,Percent Ceiling Jaccard Value,Ceiling Kappa,LLM Avg Kappa,Percent Ceiling Kappa,Ceiling Gwet AC1,LLM Avg Gwet AC1,Percent Ceiling Gwet AC1
0,10155,46,0.895,0.536,0.599,0.846,0.402,0.476,0.890,0.516,0.580,0.989,0.958,0.968
1,9948,39,0.826,0.399,0.483,0.750,0.307,0.409,0.820,0.379,0.463,0.988,0.959,0.971
2,14189,48,0.789,0.673,0.854,0.725,0.546,0.752,0.780,0.659,0.845,0.981,0.970,0.989
3,14352,41,0.700,0.569,0.813,0.523,0.413,0.790,0.683,0.548,0.802,0.964,0.954,0.990
4,14362,26,0.579,0.523,0.903,0.529,0.348,0.659,0.561,0.499,0.889,0.963,0.948,0.985
5,14363,28,0.717,0.552,0.770,0.629,0.450,0.716,0.702,0.534,0.762,0.967,0.960,0.993
6,14374,33,0.984,0.899,0.914,0.970,0.816,0.842,0.982,0.888,0.904,0.996,0.975,0.979
7,14414,41,0.729,0.685,0.940,0.583,0.543,0.930,0.708,0.663,0.937,0.955,0.953,0.998
8,14474,32,0.478,0.547,1.144,0.410,0.533,1.300,0.445,0.517,1.160,0.932,0.936,1.005
9,14499,38,0.600,0.365,0.608,0.458,0.291,0.635,0.585,0.342,0.585,0.969,0.953,0.984


In [44]:
def pair_metrics_for_kc(name_a: str, anns_a: dict[str, set[str]], name_b: str, anns_b: dict[str, set[str]], items: list[str], kc: str) -> dict:
    tp = fp = fn = tn = 0
    for item in items:
        a = kc in anns_a.get(item, set())
        b = kc in anns_b.get(item, set())
        if a and b:
            tp += 1
        elif not a and b:
            fp += 1
        elif a and not b:
            fn += 1
        else:
            tn += 1
    precision = safe_div(tp, tp + fp)
    recall = safe_div(tp, tp + fn)
    return {
        'Pair': f'{name_a} vs {name_b}',
        'KC': kc,
        'F1 Score': safe_div(2 * tp, (2 * tp) + fp + fn),
        'Binary Jaccard': safe_div(tp, tp + fp + fn),
        'Kappa': cohen_kappa_from_counts(tp, fp, fn, tn),
        "Gwet AC1": gwet_ac1_from_counts(tp, fp, fn, tn),
        'Precision': precision,
        'Recall': recall,
        'Count A': tp + fn,
        'Count B': tp + fp,
        'TP': tp,
        'FP': fp,
        'FN': fn,
        'TN': tn,
    }


per_kc_rows = []
for kc in KC_COLUMNS:
    h = pair_metrics_for_kc('Human A', human_a, 'Human B', human_b, common_items, kc)
    a = pair_metrics_for_kc('Human A', human_a, 'LLM V3', llm_v3, common_items, kc)
    b = pair_metrics_for_kc('Human B', human_b, 'LLM V3', llm_v3, common_items, kc)
    per_kc_rows.extend([h, a, b])

per_kc_df = pd.DataFrame(per_kc_rows)
per_kc_summary_rows = []
for kc in KC_COLUMNS:
    h = per_kc_df[(per_kc_df['KC'] == kc) & (per_kc_df['Pair'] == 'Human A vs Human B')].iloc[0]
    a = per_kc_df[(per_kc_df['KC'] == kc) & (per_kc_df['Pair'] == 'Human A vs LLM V3')].iloc[0]
    b = per_kc_df[(per_kc_df['KC'] == kc) & (per_kc_df['Pair'] == 'Human B vs LLM V3')].iloc[0]
    row = {'KC': kc, 'Human Count A': h['Count A'], 'Human Count B': h['Count B']}
    for metric in ['F1 Score', 'Binary Jaccard', 'Kappa', "Gwet AC1"]:
        ceiling = h[metric]
        llm_avg_metric = (a[metric] + b[metric]) / 2
        row[f'Ceiling {metric}'] = ceiling
        row[f'LLM Avg {metric}'] = llm_avg_metric
        row[f'Percent Ceiling {metric}'] = safe_div(llm_avg_metric, ceiling) if ceiling > 0 else math.nan
    per_kc_summary_rows.append(row)

per_kc_summary_df = pd.DataFrame(per_kc_summary_rows).sort_values('LLM Avg Kappa', ascending=False)
per_kc_summary_df.style.format({col: '{:.3f}' for col in per_kc_summary_df.columns if col != 'KC'})

,KC,Human Count A,Human Count B,Ceiling F1 Score,LLM Avg F1 Score,Percent Ceiling F1 Score,Ceiling Binary Jaccard,LLM Avg Binary Jaccard,Percent Ceiling Binary Jaccard,Ceiling Kappa,LLM Avg Kappa,Percent Ceiling Kappa,Ceiling Gwet AC1,LLM Avg Gwet AC1,Percent Ceiling Gwet AC1
16,ArrayIndex,31.000,23.000,0.741,0.771,1.041,0.588,0.628,1.067,0.721,0.754,1.045,0.957,0.963,1.007
17,DefFunction,10.000,9.000,0.842,0.755,0.896,0.727,0.607,0.835,0.838,0.748,0.892,0.992,0.984,0.993
12,StringIndex,36.000,29.000,0.800,0.769,0.961,0.667,0.624,0.936,0.781,0.747,0.957,0.958,0.954,0.995
14,StringEqual,15.000,14.000,0.759,0.692,0.912,0.611,0.531,0.869,0.749,0.680,0.908,0.980,0.975,0.996
6,Math%,4.000,3.000,0.286,0.679,2.375,0.167,0.542,3.250,0.279,0.675,2.419,0.986,0.993,1.007
2,While,3.000,3.000,1.000,0.667,0.667,1.000,0.500,0.500,1.000,0.663,0.663,1.000,0.992,0.992
11,StringConcat,20.000,22.000,0.714,0.675,0.946,0.556,0.510,0.918,0.697,0.659,0.945,0.964,0.964,1.000
5,Math+-*/,16.000,17.000,0.606,0.667,1.101,0.435,0.502,1.155,0.588,0.653,1.110,0.962,0.969,1.008
9,LogicBoolean,13.000,12.000,0.560,0.616,1.101,0.389,0.453,1.164,0.545,0.602,1.105,0.968,0.970,1.001
13,StringLen,12.000,13.000,0.560,0.615,1.097,0.389,0.446,1.146,0.545,0.598,1.098,0.968,0.963,0.995


In [45]:
overall_df.to_csv(OUTPUT_DIR / 'overall_metrics.csv', index=False)
ceiling_df.to_csv(OUTPUT_DIR / 'ceiling_metrics.csv', index=False)
student_df.to_csv(OUTPUT_DIR / 'per_student_metrics.csv', index=False)
per_kc_df.to_csv(OUTPUT_DIR / 'per_kc_pair_metrics.csv', index=False)
per_kc_summary_df.to_csv(OUTPUT_DIR / 'per_kc_ceiling_metrics.csv', index=False)

summary_payload = {
    'human_ceiling': human_ceiling,
    'llm_vs_human_a': llm_vs_a,
    'llm_vs_human_b': llm_vs_b,
    'llm_average_vs_humans': llm_avg,
    'ceiling': ceiling_df.to_dict(orient='records'),
    'common_items': len(common_items),
    'kc_columns': KC_COLUMNS,
    'include_empty_jaccard': INCLUDE_EMPTY_JACCARD,
}

with (OUTPUT_DIR / 'v3_prompt_eval_summary.json').open('w', encoding='utf-8') as f:
    json.dump(summary_payload, f, indent=2)

print(f'Saved results to {OUTPUT_DIR}')
print('\n=== Human Ceiling vs LLM Average ===')
for _, row in ceiling_df.iterrows():
    print(
        f"{row['Metric']:<14} "
        f"Human ceiling={row['Human Ceiling']:.3f} | "
        f"LLM avg={row['LLM Avg vs Humans']:.3f} | "
        f"Gap={row['Gap to Ceiling']:.3f} | "
        f"Percent ceiling={row['Percent of Ceiling']:.1%}"
    )

Saved results to /mnt/d/Projects/kintsugi/results/human_validation/v3_prompt_eval_results

=== Human Ceiling vs LLM Average ===
F1 Score       Human ceiling=0.758 | LLM avg=0.618 | Gap=0.140 | Percent ceiling=81.5%
Jaccard Value  Human ceiling=0.668 | LLM avg=0.501 | Gap=0.167 | Percent ceiling=75.1%
Kappa          Human ceiling=0.745 | LLM avg=0.598 | Gap=0.147 | Percent ceiling=80.3%
Gwet AC1       Human ceiling=0.971 | LLM avg=0.957 | Gap=0.014 | Percent ceiling=98.5%
